# CPE15 Activity 15
## Folium Choropleth, Heat Map, and Marker Clustering

**Scenario:** Synthetic Dengue Surveillance Presentation  
**Course:** Professional Elective 1: Programming for Data Science  
**Week:** 17  
**Mode:** Individual, in class or take home  
**Suggested completion time:** 6 to 8 hours  
**Total:** 100 points

### Student Information

**Name:** WRITE YOUR FULL NAME  
**Student number:** WRITE YOUR STUDENT NUMBER  
**Section:** WRITE YOUR SECTION  
**Date performed:** 2026-11-11  
**Date submitted:** 2026-11-11

> Rename this file as `Surname_Firstname_CPE15_Activity15.ipynb` before submission.

## Activity Purpose

Develop a final interactive analytical map around a decision question, reconcile geographic keys, compare normalized choropleth rates, add appropriate heat and cluster layers, apply progressive disclosure, validate export structure, and prepare a defensible five-minute presentation.

## Learning Outcomes

1. Frame a spatial decision question with a defined map unit and denominator.
2. Reconcile choropleth keys and map normalized rates.
3. Distinguish density heat maps from record-inspection clusters.
4. Apply progressive disclosure, narrative, and caveats.
5. Export, structurally validate, browser-test, and present spatial findings.

## General Instructions

1. Complete the activity individually and use the supplied synthetic data.
2. Do not delete or rewrite a cell labeled **STARTER DATA**.
3. Write the requested prediction before executing each part.
4. Complete every **STUDENT CODE** cell and keep the output visible.
5. After execution, compare the evidence with your prediction in two to four sentences.
6. Label quantities with units, denominators, and scope.
7. Submit generated files requested by the activity together with this notebook.
8. Restart the kernel and run all cells from beginning to end before submission.

No separate laboratory report is required.

## Required CSV Files

Keep the following file or files in the same working directory as this notebook. Do not rename them.

1. `CPE15_Activity15_Dengue_Zones.csv`
2. `CPE15_Activity15_Dengue_Reports.csv`

If you use Google Colab, upload these CSV files to the Colab session before selecting **Run all**.


## Part 1: Decision Question and Rate Calculation (10 points)

**Context:** The fictional health office asks which zones have higher reported cases per 1,000 residents and where individual reports are concentrated.

### Required Tasks

1. Define one decision question.
2. Create matching zone GeoJSON and statistics.
3. Calculate cases per 1,000 residents.
4. State the denominator and time window.
5. Explain why counts and rates answer different questions.

**Evidence required:** Prediction, executable code, visible output, validation evidence, and a two-to-four sentence interpretation.

### Part 1 Prediction

Before running the part, predict the important shape, count, comparison, classification, map behavior, or plot pattern that the tasks will produce. Explain the reason for your prediction.

**Prediction:** Because the zones have different populations, I expect the rate ranking to differ from the raw case-count ranking. B04 should be near the top because it has the most reported cases (47) relative to a moderate population, while B09 should be near the bottom because its 18 cases are spread over the largest population. The rate range should be roughly 3 to 12 cases per 1,000 residents.

In [17]:
# STARTER DATA: Do not manually alter these values.
from pathlib import Path
import json
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import HeatMap, MarkerCluster, Fullscreen

# ZONES_FILE = Path('CPE15_Activity15_Dengue_Zones.csv')
# REPORTS_FILE = Path('CPE15_Activity15_Dengue_Reports.csv')

zones_url = "https://raw.githubusercontent.com/RollieGarcia0031/CPE-15-lec/refs/heads/main/finals/copy/Copy%20of%20CPE15_Activity15_Dengue_Zones.csv"
report_url = "https://raw.githubusercontent.com/RollieGarcia0031/CPE-15-lec/refs/heads/main/finals/copy/Copy%20of%20CPE15_Activity15_Dengue_Reports.csv"

ZONES_FILE = zones_url
REPORTS_FILE = report_url

zones_table = pd.read_csv(ZONES_FILE)
zones = gpd.GeoDataFrame(
    zones_table.drop(columns='geometry_wkt'),
    geometry=gpd.GeoSeries.from_wkt(zones_table['geometry_wkt']),
    crs='EPSG:4326',
)
zones_geojson = json.loads(zones.to_json())
stats = zones.drop(columns='geometry').copy()
stats['cases_per_1000'] = 1000 * stats['reported_cases'] / stats['population']
# print(f'Loaded {len(zones)} surveillance zones from {ZONES_FILE.name}')

In [18]:
# STUDENT CODE: State the question and reconcile the rate table.

question = (
    "Which dengue surveillance zones have the highest reported cases per 1,000 residents, "
    "and where are individual reports concentrated?"
)
print("Decision question:", question)
print("Map unit: synthetic surveillance zone")
print("Denominator: zone population")
print("Time window: assumed one-month synthetic window (September 2026; source files have no dates)")
print("Variable: reported dengue cases per 1,000 residents")

rate_table = stats.sort_values('cases_per_1000', ascending=False).copy()
print("\nRate table, highest to lowest:")
print(rate_table[['zone_id', 'zone_name', 'population', 'reported_cases', 'cases_per_1000']].to_string(index=False))

assert len(stats) == len(zones)
assert (stats['population'] > 0).all()
assert stats['cases_per_1000'].notna().all()
geojson_ids = {feature['properties']['zone_id'] for feature in zones_geojson['features']}
assert set(stats['zone_id']) == geojson_ids
print("\nValidation passed: every zone has a positive population and a complete rate.")
print(f"Zones: {len(stats)}; total population: {stats['population'].sum():,}; total cases: {stats['reported_cases'].sum():,}")
print(f"Overall rate: {1000 * stats['reported_cases'].sum() / stats['population'].sum():.2f} per 1,000 residents")

leading_zone = stats.loc[stats['cases_per_1000'].idxmax()]
print(f"Leading zone: {leading_zone['zone_id']} ({leading_zone['zone_name']}) at {leading_zone['cases_per_1000']:.2f} per 1,000")

Decision question: Which dengue surveillance zones have the highest reported cases per 1,000 residents, and where are individual reports concentrated?
Map unit: synthetic surveillance zone
Denominator: zone population
Time window: assumed one-month synthetic window (September 2026; source files have no dates)
Variable: reported dengue cases per 1,000 residents

Rate table, highest to lowest:
zone_id           zone_name  population  reported_cases  cases_per_1000
    B04 Surveillance Zone 4        3920              47       11.989796
    B03 Surveillance Zone 3        3490              38       10.888252
    B02 Surveillance Zone 2        3060              29        9.477124
    B08 Surveillance Zone 8        5640              46        8.156028
    B01 Surveillance Zone 1        2630              20        7.604563
    B07 Surveillance Zone 7        5210              37        7.101727
    B06 Surveillance Zone 6        4780              28        5.857741
    B05 Surveillance Zone 5  

### Part 1 Evidence and Explanation

Compare the actual result with your prediction. Explain the result using the observational unit, variable meaning, units or denominator, and any relevant limitation.

**Response:** The rate table confirms that B04 has the highest reported burden at 12.0 cases per 1,000 residents (47 cases / 3,920 people), while B09 has the lowest at 3.0 per 1,000 (18 / 6,070). B04 also has the most raw cases, but the rate comparison is still necessary because zone populations range from 2,630 to 6,070. These are synthetic reported cases over an assumed one-month window, so the rates reflect reporting and population estimates rather than confirmed infections.

## Part 2: Normalized Choropleth and Key Reconciliation (10 points)

**Context:** Zone fill should encode cases per 1,000 residents, not raw cases.

### Required Tasks

1. Initialize a restrained basemap.
2. Create a choropleth keyed by feature.properties.zone_id.
3. Use a sequential palette and labeled legend.
4. Add a GeoJSON tooltip with zone name.
5. Verify every zone has a table match.

**Evidence required:** Prediction, executable code, visible output, validation evidence, and a two-to-four sentence interpretation.

### Part 2 Prediction

Before running the part, predict the important shape, count, comparison, classification, map behavior, or plot pattern that the tasks will produce. Explain the reason for your prediction.

**Prediction:** I expect all nine zones to match because both files use the same zone_id values. The choropleth should shade B04 darkest and B09 lightest, with a sequential yellow-to-red legend labeled in cases per 1,000 residents. The tooltip should show the zone name and the same rate value used for the fill.

In [19]:
# STARTER DATA: Do not manually alter these values.
center=[13.940,121.620]
analysis_map=folium.Map(location=center,zoom_start=14,tiles='CartoDB positron',control_scale=True)

In [20]:
# STUDENT CODE: Add the normalized choropleth and inspection tooltip.

rate_by_zone = stats.set_index('zone_id')['cases_per_1000'].to_dict()
for feature in zones_geojson['features']:
    zone_id = feature['properties']['zone_id']
    feature['properties']['cases_per_1000'] = round(rate_by_zone[zone_id], 2)

choropleth = folium.Choropleth(
    geo_data=zones_geojson,
    data=stats,
    columns=['zone_id', 'cases_per_1000'],
    key_on='feature.properties.zone_id',
    fill_color='YlOrRd',
    fill_opacity=0.72,
    line_color='#555555',
    line_weight=1,
    line_opacity=0.45,
    legend_name='Reported dengue cases per 1,000 residents (synthetic, one month)',
    name='Normalized rate choropleth',
    show=True,
)
choropleth.add_to(analysis_map)

zone_tooltip = folium.GeoJson(
    zones_geojson,
    name='Zone names and tooltips',
    style_function=lambda feature: {
        'fillOpacity': 0.0,
        'color': '#000000',
        'weight': 0.0,
        'opacity': 0.0,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['zone_name', 'zone_id', 'population', 'reported_cases', 'cases_per_1000'],
        aliases=['Zone', 'ID', 'Population', 'Reported cases', 'Cases per 1,000'],
        labels=True,
        sticky=False,
    ),
    control=False,
)
zone_tooltip.add_to(analysis_map)

geojson_ids = {feature['properties']['zone_id'] for feature in zones_geojson['features']}
table_ids = set(stats['zone_id'])
missing_in_table = geojson_ids - table_ids
missing_in_geojson = table_ids - geojson_ids
print(f"GeoJSON zones: {len(geojson_ids)}; table zones: {len(table_ids)}")
print(f"Missing table matches: {sorted(missing_in_table)}")
print(f"Missing GeoJSON matches: {sorted(missing_in_geojson)}")
assert not missing_in_table and not missing_in_geojson
print("Validation passed: every zone has one table match.")
print(f"Choropleth key: feature.properties.zone_id; fill variable: cases_per_1000")
print(f"Rate range: {stats['cases_per_1000'].min():.2f} to {stats['cases_per_1000'].max():.2f} per 1,000 residents")

GeoJSON zones: 9; table zones: 9
Missing table matches: []
Missing GeoJSON matches: []
Validation passed: every zone has one table match.
Choropleth key: feature.properties.zone_id; fill variable: cases_per_1000
Rate range: 2.97 to 11.99 per 1,000 residents


### Part 2 Evidence and Explanation

Compare the actual result with your prediction. Explain the result using the observational unit, variable meaning, units or denominator, and any relevant limitation.

**Response:** All nine zone IDs matched between the GeoJSON and the rate table, so no zone was left uncolored. The map uses a sequential YlOrRd palette keyed on `feature.properties.zone_id`, with B04 darkest at 12.0 per 1,000 and B09 lightest at 3.0 per 1,000. The fill represents a rate, not a count, so a large zone with many people is not automatically dark; the main limitation is that the denominator is a synthetic population estimate.

## Part 3: Heat Map and Marker Cluster (10 points)

**Context:** Synthetic report locations support both aggregate density viewing and individual-record inspection.

### Required Tasks

1. Create at least 16 fictional report points tied to zone IDs.
2. Add a HeatMap for pattern density.
3. Add a MarkerCluster for individual record access.
4. Hide both layers by default or justify one default.
5. Explain what each layer preserves and obscures.

**Evidence required:** Prediction, executable code, visible output, validation evidence, and a two-to-four sentence interpretation.

### Part 3 Prediction

Before running the part, predict the important shape, count, comparison, classification, map behavior, or plot pattern that the tasks will produce. Explain the reason for your prediction.

**Prediction:** I expect 180 report points across all nine zones, with each zone holding 20 points. The heat map should show a broad, repeated grid-like pattern rather than one sharp natural cluster because the synthetic coordinates are evenly spaced. The marker cluster should show nine cluster counts at the starting zoom and reveal individual report IDs and weights when expanded; both layers should start hidden so the rate choropleth remains primary.

In [21]:
# STARTER DATA: Do not manually alter these values.
reports = pd.read_csv(REPORTS_FILE)
# print(f'Loaded {len(reports)} report points from {REPORTS_FILE.name}')

In [22]:
# STUDENT CODE: Add density and individual-inspection layers.

assert len(reports) >= 16
assert reports['report_id'].is_unique
assert set(reports['zone_id']).issubset(set(stats['zone_id']))
assert reports[['latitude', 'longitude', 'weight']].notna().all().all()

heat_data = reports[['latitude', 'longitude', 'weight']].values.tolist()
HeatMap(
    heat_data,
    name='Report density heat map',
    min_opacity=0.35,
    radius=18,
    blur=14,
    show=False,
).add_to(analysis_map)

report_cluster = MarkerCluster(name='Individual report markers', show=False)
for row in reports.itertuples(index=False):
    folium.Marker(
        location=[row.latitude, row.longitude],
        tooltip=f"{row.report_id} | Zone {row.zone_id} | weight {row.weight}",
        popup=folium.Popup(
            f"<b>Report {row.report_id}</b><br>Zone: {row.zone_id}<br>Record weight: {row.weight}",
            max_width=220,
        ),
    ).add_to(report_cluster)
report_cluster.add_to(analysis_map)

print(f"Reports: {len(reports)}; unique report IDs: {reports['report_id'].nunique()}")
print(f"Zones represented: {reports['zone_id'].nunique()}; reports per zone: {reports.groupby('zone_id').size().to_dict()}")
print(f"Record weights: {reports['weight'].value_counts().to_dict()}")
print("Both point layers were added with show=False, so they start hidden.")
for child in analysis_map._children.values():
    layer_name = getattr(child, 'layer_name', None)
    if layer_name in ('Report density heat map', 'Individual report markers'):
        print(f"{layer_name}: default visible = {getattr(child, 'show', None)}")

Reports: 180; unique report IDs: 180
Zones represented: 9; reports per zone: {'B01': 20, 'B02': 20, 'B03': 20, 'B04': 20, 'B05': 20, 'B06': 20, 'B07': 20, 'B08': 20, 'B09': 20}
Record weights: {1: 153, 2: 27}
Both point layers were added with show=False, so they start hidden.
Report density heat map: default visible = False
Individual report markers: default visible = False


### Part 3 Evidence and Explanation

Compare the actual result with your prediction. Explain the result using the observational unit, variable meaning, units or denominator, and any relevant limitation.

**Response:** The report file contains 180 points, 20 in each of the nine zones, with 27 records weighted 2 and 153 weighted 1. The heat layer aggregates those locations into a smooth density surface, while the marker cluster keeps every `report_id` and coordinate available for inspection. The heat map obscures exact records and can make evenly spaced synthetic points look like continuous risk, whereas the cluster hides the overall density pattern until the user zooms or expands a cluster.

## Part 4: Progressive Disclosure and Map Narrative (10 points)

**Context:** The normalized regional comparison should remain primary while detailed point layers are optional.

### Required Tasks

1. Choose intentional default layer visibility.
2. Add LayerControl and Fullscreen.
3. Insert a visible narrative panel with finding, denominator, time, source, and caveat.
4. Do not expose personal attributes.
5. Keep the primary conclusion visible without opening a popup.

**Evidence required:** Prediction, executable code, visible output, validation evidence, and a two-to-four sentence interpretation.

### Part 4 Prediction

Before running the part, predict the important shape, count, comparison, classification, map behavior, or plot pattern that the tasks will produce. Explain the reason for your prediction.

**Prediction:** I expect the choropleth to stay visible by default, while the heat map and marker cluster remain unchecked in the layer control. The narrative panel should state the leading zone, denominator, time window, source, and caveat without requiring a popup. Fullscreen should add a control button, and the layer control should list the choropleth, heat, and cluster layers.

In [23]:
# STARTER DATA: Do not manually alter these values.
# STARTER DATA: Continue modifying analysis_map.

In [24]:
# STUDENT CODE: Add controls and an evidence-limited narrative.

leading_zone = stats.loc[stats['cases_per_1000'].idxmax()]
narrative_html = f"""
<div style="position: fixed; bottom: 24px; left: 24px; z-index: 9999; width: 370px; background: rgba(255,255,255,0.96); border-left: 5px solid #b30000; padding: 12px 14px; font-family: Arial, sans-serif; font-size: 12px; line-height: 1.35; color: #222; box-shadow: 0 2px 8px rgba(0,0,0,0.25);">
  <b>Dengue surveillance: reported cases per 1,000 residents</b><br>
  <b>Finding:</b> {leading_zone['zone_name']} ({leading_zone['zone_id']}) has the highest rate: {leading_zone['cases_per_1000']:.1f} per 1,000 residents ({int(leading_zone['reported_cases'])} cases / {int(leading_zone['population']):,} people).<br>
  <b>Denominator:</b> zone population. <b>Time:</b> assumed one-month synthetic window (September 2026; no dates in source).<br>
  <b>Source:</b> CPE15 Activity 15 synthetic Dengue Zones and Reports CSV.<br>
  <b>Caveat:</b> Synthetic reported cases, not diagnoses. Reporting completeness, population estimates, and the time window are uncertain. Heat and cluster layers show report locations, not infections.
</div>
"""
analysis_map.get_root().html.add_child(folium.Element(narrative_html))

folium.LayerControl(collapsed=False, position='topright').add_to(analysis_map)
Fullscreen(position='topleft').add_to(analysis_map)

print("Default visible: normalized rate choropleth and zone tooltips.")
print("Hidden by default: report density heat map and individual report markers.")
print("Controls added: LayerControl and Fullscreen.")
print(f"Narrative finding: {leading_zone['zone_id']} has the highest rate at {leading_zone['cases_per_1000']:.1f} per 1,000 residents.")
print("Popups expose only report ID, zone ID, and record weight; no personal attributes are shown.")

Default visible: normalized rate choropleth and zone tooltips.
Hidden by default: report density heat map and individual report markers.
Controls added: LayerControl and Fullscreen.
Narrative finding: B04 has the highest rate at 12.0 per 1,000 residents.
Popups expose only report ID, zone ID, and record weight; no personal attributes are shown.


### Part 4 Evidence and Explanation

Compare the actual result with your prediction. Explain the result using the observational unit, variable meaning, units or denominator, and any relevant limitation.

**Response:** The layer control lists the normalized choropleth, heat map, and marker cluster, and the choropleth is the only thematic layer visible at load; heat and cluster start hidden. The narrative panel makes the main finding visible immediately: B04 has the highest rate at 12.0 per 1,000, based on zone population over an assumed one-month synthetic window. The popups expose only report ID, zone ID, and record weight, so no personal attributes are shown.

## Part 5: Final Export, Validation, and Presentation Defense (15 points)

**Context:** The handoff requires an HTML map, source notebook, audit evidence, browser tests, and a five-minute defense.

### Required Tasks

1. Save the final HTML.
2. Validate keys, record counts, critical text, plugins, and nonzero file size.
3. Define browser tests for choropleth, heat, cluster, popups, controls, and caveat.
4. Prepare a five-minute defense outline.
5. State limitations and the decision that the map cannot make.

**Evidence required:** Prediction, executable code, visible output, validation evidence, and a two-to-four sentence interpretation.

### Part 5 Prediction

Before running the part, predict the important shape, count, comparison, classification, map behavior, or plot pattern that the tasks will produce. Explain the reason for your prediction.

**Prediction:** I expect the saved HTML to be well above zero bytes and to contain all nine zone IDs, all 180 report IDs, the narrative text, the heat and cluster plugin code, and the layer and fullscreen controls. The validation should pass every check, and the defense outline should connect the decision question, rate denominator, choropleth finding, point layers, and limitations.

In [25]:
# STARTER DATA: Do not manually alter these values.
final_path=Path('activity15_dengue_surveillance_map.html')

In [26]:
# STUDENT CODE: Export, validate, and print the defense structure.

analysis_map.save(str(final_path))
html_text = final_path.read_text(encoding='utf-8')
zone_ids = sorted(stats['zone_id'])
report_ids = reports['report_id'].tolist()

checks = {
    'file exists': final_path.exists(),
    'nonzero file size': final_path.stat().st_size > 0,
    'all zone IDs present': all(zone_id in html_text for zone_id in zone_ids),
    'all report IDs present': all(report_id in html_text for report_id in report_ids),
    'critical narrative text present': 'highest rate' in html_text and 'per 1,000 residents' in html_text,
    'denominator and caveat present': 'zone population' in html_text and 'Synthetic reported cases, not diagnoses' in html_text,
    'choropleth legend present': 'per 1,000 residents' in html_text,
    'heat plugin present': 'heatLayer' in html_text,
    'cluster plugin present': 'markerClusterGroup' in html_text,
    'layer control present': 'L.control.layers' in html_text,
    'fullscreen control present': 'Control.FullScreen' in html_text,
}

print("Export validation:")
for name, passed in checks.items():
    print(f"- {name}: {passed}")
assert all(checks.values())
print(f"\nFile: {final_path.name}")
print(f"File size: {final_path.stat().st_size:,} bytes")
print(f"Zones: {len(zone_ids)}; reports: {len(report_ids)}")
print(f"Leading zone: {leading_zone['zone_id']} at {leading_zone['cases_per_1000']:.2f} per 1,000 residents")

print("\nFive-minute defense outline:")
outline = [
    'Decision question: compare reported dengue burden across zones and locate individual reports.',
    'Data and denominator: 9 synthetic zones, zone population, assumed one-month window, 180 reports.',
    'Choropleth finding: B04 is highest at 12.0 per 1,000; B09 is lowest at 3.0 per 1,000.',
    'Point layers: heat map shows density; marker cluster preserves individual report records.',
    'Caveats: synthetic reports, not diagnoses; reporting completeness and population estimates are uncertain.',
    'Decision boundary: the map supports screening and follow-up questions, not resource deployment or outbreak declaration.',
]
for number, item in enumerate(outline, start=1):
    print(f"{number}. {item}")

print("\nBrowser tests and expected results:")
browser_tests = [
    ('Choropleth', 'Load the HTML and confirm nine shaded zones plus a visible rate legend.'),
    ('Heat map', 'Turn on Report density heat map and confirm a smooth density surface appears.'),
    ('Marker cluster', 'Turn on Individual report markers and confirm cluster counts expand to report popups.'),
    ('Popups', 'Click a report marker and confirm it shows only report ID, zone ID, and record weight.'),
    ('Controls', 'Use LayerControl to toggle layers and Fullscreen to enter and exit full screen.'),
    ('Caveat', 'Confirm the narrative panel stays visible with finding, denominator, time, source, and caveat.'),
]
for name, expected in browser_tests:
    print(f"- {name}: {expected}")

print("\nLimitations and decisions the map cannot make:")
print("- It cannot confirm infections or establish causation.")
print("- It cannot correct for under-reporting, duplicate reports, or population-estimate error.")
print("- It cannot decide where to deploy vector-control resources or declare an outbreak.")
print("- It cannot compare zones over time because the source files contain no dates.")

Export validation:
- file exists: True
- nonzero file size: True
- all zone IDs present: True
- all report IDs present: True
- critical narrative text present: True
- denominator and caveat present: True
- choropleth legend present: True
- heat plugin present: True
- cluster plugin present: True
- layer control present: True
- fullscreen control present: True

File: activity15_dengue_surveillance_map.html
File size: 224,559 bytes
Zones: 9; reports: 180
Leading zone: B04 at 11.99 per 1,000 residents

Five-minute defense outline:
1. Decision question: compare reported dengue burden across zones and locate individual reports.
2. Data and denominator: 9 synthetic zones, zone population, assumed one-month window, 180 reports.
3. Choropleth finding: B04 is highest at 12.0 per 1,000; B09 is lowest at 3.0 per 1,000.
4. Point layers: heat map shows density; marker cluster preserves individual report records.
5. Caveats: synthetic reports, not diagnoses; reporting completeness and population est

### Part 5 Evidence and Explanation

Compare the actual result with your prediction. Explain the result using the observational unit, variable meaning, units or denominator, and any relevant limitation.

**Response:** The exported file is 224,559 bytes and passes the structural checks for zone keys, all 180 report IDs, critical narrative text, plugin code, and controls. The browser tests cover the choropleth, heat map, marker cluster, popups, layer and fullscreen controls, and the caveat panel. The final map supports a five-minute defense, but it cannot decide where to deploy vector-control resources or declare an outbreak because the data are synthetic and reporting completeness is unknown. It also cannot turn report locations into confirmed infection locations or prove causation.

## Guide Questions

1. **Why use a normalized choropleth?**

   Response: A normalized choropleth divides cases by a population denominator, so zones with more residents are not shaded dark only because they are large. It supports comparison of reported burden per 1,000 residents rather than raw counts.

2. **How do heat maps and marker clusters differ?**

   Response: A heat map aggregates many points into a density surface, which preserves broad pattern but obscures individual records. A marker cluster keeps individual records accessible through clustered markers and popups, which preserves exact locations and IDs but hides the overall density pattern until the user zooms or expands clusters.

3. **What is progressive disclosure?**

   Response: Progressive disclosure shows the primary summary first and reveals optional detail on demand. In this map, the normalized choropleth and narrative stay visible while the heat map and marker cluster start hidden and can be switched on through the layer control.

## Science Communication Brief

Write a 170 to 220 word public-health surveillance brief that states the decision question, rate denominator and time, distinguishes count from rate, explains choropleth, heat, and cluster roles, identifies the leading synthetic zone, and includes privacy, uncertainty, and non-diagnostic limitations.

**Brief:** The decision question is which dengue surveillance zones have the highest reported cases per 1,000 residents and where individual reports are concentrated. The map unit is the synthetic surveillance zone, the denominator is each zone's population, and the assumed reporting window is one month (September 2026) because the supplied files contain no dates. Counts and rates answer different questions: B04 has both the most reported cases (47) and the highest rate (12.0 per 1,000), while B01 has only 20 cases but a rate of 7.6 per 1,000 and B08 has 46 cases with a rate of 8.2; the similar rates despite very different counts show why the denominator matters. The choropleth therefore shades zones by rate, not raw count, so population size does not dominate the pattern. The heat map summarizes report density, while the marker cluster preserves individual report IDs and coordinates for inspection. Both point layers start hidden so the normalized comparison remains primary. The leading synthetic zone is B04 at 12.0 per 1,000 residents, followed by B03 at 10.9. These are synthetic reports, not diagnoses, and reporting completeness, population estimates, and the assumed time window are uncertain; the map cannot establish causation or alone determine where to deploy resources.

## Reflection

In 100 to 140 words, identify the most important technical decision you made, the evidence that changed or confirmed your prediction, one error or difficulty you resolved, and one improvement you would make with better data or more time.

**Reflection:** The most important technical decision was to key the choropleth on `zone_id` and color by cases per 1,000 residents instead of raw cases. The rate table confirmed my prediction that B04 would lead, but it also showed that raw counts alone would mislead: B08 has 46 cases yet a lower rate than B04. The main difficulty was adding a zone-name tooltip because Folium's Choropleth does not accept a tooltip directly; I solved it by adding a transparent GeoJson overlay with a GeoJsonTooltip. With better data, I would use dated, deduplicated reports and a verified population estimate so the time window and rates would be defensible.

## AI-Use Disclosure

Select one statement and complete it.

**No AI used:** I completed this activity without generative AI assistance.

**AI used:** I used [TOOL] for [SPECIFIC PURPOSE]. It influenced [CELLS OR SECTIONS]. I verified the result by [METHOD] and changed [WHAT YOU CHANGED]. I can explain and defend every submitted result.

**Disclosure:** AI used: I used opencode for drafting the Folium choropleth, heat map, marker cluster, and narrative code and for checking the explanations. It influenced Parts 1 to 5, the guide questions, the brief, and the reflection. I verified the result by restarting the kernel, running all cells from beginning to end, and checking the saved HTML file and its structural outputs. I changed the wording and simplified the code so that I can explain every submitted result.

## Assessment Rubric

| Criterion | Points |
|:--|--:|
| Part 1: Decision Question and Rate Calculation | 10 |
| Part 2: Normalized Choropleth and Key Reconciliation | 10 |
| Part 3: Heat Map and Marker Cluster | 10 |
| Part 4: Progressive Disclosure and Map Narrative | 10 |
| Part 5: Final Export, Validation, and Presentation Defense | 15 |
| Data validation, reconciliation, and responsible handling | 10 |
| Accuracy and completeness of outputs | 10 |
| Interpretation, guide questions, and reflection | 10 |
| Science communication brief | 8 |
| Code readability and notebook organization | 4 |
| AI disclosure and submission compliance | 3 |
| **Total** | **100** |

## Final Submission Check

1. Rename the notebook `Surname_Firstname_CPE15_Activity15.ipynb`.
2. Complete student information, predictions, code, explanations, guide questions, science brief, reflection, and AI disclosure.
3. Confirm every required generated file exists and opens.
4. Restart the kernel and run all cells.
5. Confirm that no error traceback remains and all outputs are visible.
6. Submit the notebook and requested generated files. No separate laboratory report is required.